In [1]:
import copy
import json
import math
import numpy as np
import os
import pandas as pd

from scipy.spatial.distance import cosine

import IPython.core.display


In [2]:
# Get the root_path for this jupyter notebook repo.
repo_path = os.path.dirname(os.path.abspath(os.getcwd()))
pc_data_path = os.path.join(
    repo_path, 'files', 'poggio-civitate',
)
# this is the path to the main dataset, with all of the cataloged objects.
object_data_path = os.path.join(
    pc_data_path, 'pc-catalog-objs-embeddings.csv',
)
output_html = os.path.join(
    pc_data_path, 'pc-catalog-objs-embeddings.html',
)
net_output_html = os.path.join(
    pc_data_path, 'pc-catalog-objs-embeddings-net.html',
)


df_all = pd.read_csv(object_data_path)
print(f'We have {len(df_all.index)} records of object data')
print(df_all.columns.tolist())

We have 13541 records of object data
['Unnamed: 0', 'uuid', 'label', 'path', 'item_class__slug', 'project__label', 'project__uuid', 'geo_source__path', 'geo_source__uuid', 'chrono_source__path', 'chrono_source__uuid', 'latitude', 'longitude', 'earliest', 'latest', 'str_for_embedding', 'embedding', 'Unnamed: 0.5', 'Unnamed: 0.4', 'Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'thumbnail_uri']


In [3]:
def get_valid_data_sample(df_all, n=100):
    valid_index = (
        ~df_all['embedding'].isnull()
        & ~df_all['latitude'].isnull()
        & ~df_all['longitude'].isnull()
        & ~df_all['uuid'].isnull()
    )
    df_sample = df_all[valid_index].sample(n=n, random_state=1).copy()
    df_sample.reset_index(drop=True, inplace=True)
    df_sample['embedding'] = df_sample['embedding'].apply(lambda x: json.loads(x))
    return df_sample

df_sample = get_valid_data_sample(df_all, n=200)
print(f'We have {len(df_sample.index)} valid sample records of object data')

We have 200 valid sample records of object data


In [4]:
def get_difference_between_embeddings(e1, e2):
    """Get a difference distance between two embeddings"""
    v1 = np.array(e1)
    v2 = np.array(e2)
    return cosine(v1, v2)

def get_geodist(lat_1, lon_1, lat_2, lon_2, max_dist=None):
    lat_sqr = (lat_1 - lat_2)**2
    lon_sqr = (lon_1 - lon_2)**2
    dist = math.sqrt((lat_sqr + lon_sqr))
    if not max_dist:
        return dist
    # Return the distance, normalized as a fraction of the
    # max_distance
    return dist / max_dist

distance_weights = {
    'vector_dist': 0.8,
    'geo_normal_dist': 0.2,
}


In [5]:
def make_raw_distance_matrix(df_sample):
    done_pairs = []
    # make the maximum geo distance to normalize the geo distances
    max_lat = df_sample['latitude'].max()
    max_lon = df_sample['longitude'].max()
    min_lat = df_sample['latitude'].min()
    min_lon = df_sample['longitude'].min()
    max_geo_dist = get_geodist(lat_1=max_lat, lon_1=max_lon, lat_2=min_lat, lon_2=min_lon)
    rows = []
    for _, row in df_sample.iterrows():
        uuid = row['uuid']
        uuid_g = str(row['uuid']).replace('-', '_')
        other_index = ~(df_sample['uuid'] == uuid)
        for _, orow in df_sample[other_index].iterrows():
            o_uuid_g = str(orow['uuid']).replace('-', '_')
            o_uuid = orow['uuid']
            act_pair = [uuid, o_uuid]
            act_pair.sort()
            if act_pair in done_pairs:
                continue
            done_pairs.append(act_pair)
            cos_dist = get_difference_between_embeddings(row['embedding'], orow['embedding'])
            act_dict = {
                'group': row['item_class__slug'].split('-cat-')[-1],
                'source_label': row['label'],
                'target_label': orow['label'],
                'source_uuid': uuid,
                'target_uuid': o_uuid,
                'source': uuid_g, # use the graph version with '_' replacing '-'
                'target': o_uuid_g, # use the graph version with '_' replacing '-'
                'vector_dist': cos_dist,
                'geo_normal_dist': get_geodist(
                    lat_1=row['latitude'], 
                    lon_1=row['longitude'], 
                    lat_2=orow['latitude'],  
                    lon_2=orow['longitude'],  
                    max_dist=max_geo_dist,
                )
            }
            rows.append(act_dict)
    df_raw_dist = pd.DataFrame(data=rows)
    return df_raw_dist
            
    
def make_weighted_similaries(df_raw_dist, distance_weights=distance_weights):
    df_sim = df_raw_dist.copy()
    df_sim['weight'] = float(0.0)
    for i, row in df_sim.iterrows():
        distance = 0
        for key, weight in distance_weights.items():
            distance += (row.get(key, 0) * weight)
        df_sim.at[i, 'weight'] = (1 - distance) * 100
    return df_sim

# Make raw distance dataframe
df_raw_dist = make_raw_distance_matrix(df_sample)
df_raw_dist.head(50)
# Make weighted similarities (similarity = 1 - distance)
df_sim = make_weighted_similaries(df_raw_dist, distance_weights=distance_weights)
df_sim.head(50)
            

,group,source_label,target_label,source_uuid,target_uuid,source,target,vector_dist,geo_normal_dist,weight
0,object,PC 19960097,PC 19770127,30d266ad-d438-495d-304f-db750bb1066a,99c092da-cab5-4ae8-3857-2df8c7ddf30f,30d266ad_d438_495d_304f_db750bb1066a,99c092da_cab5_4ae8_3857_2df8c7ddf30f,0.581649,0.004885,53.370351
1,object,PC 19960097,PC 20170071,30d266ad-d438-495d-304f-db750bb1066a,8b04da4d-2ad6-4495-8913-d89d1ed3a5d2,30d266ad_d438_495d_304f_db750bb1066a,8b04da4d_2ad6_4495_8913_d89d1ed3a5d2,0.434550,0.089322,63.449586
2,object,PC 19960097,PC 19670353,30d266ad-d438-495d-304f-db750bb1066a,a77d8e49-d260-4fe6-6ec6-4637b1245230,30d266ad_d438_495d_304f_db750bb1066a,a77d8e49_d260_4fe6_6ec6_4637b1245230,0.598336,0.060559,50.921901
3,object,PC 19960097,VdM20060216,30d266ad-d438-495d-304f-db750bb1066a,d65755e6-49e7-41b4-c41f-ac40f6d9142b,30d266ad_d438_495d_304f_db750bb1066a,d65755e6_49e7_41b4_c41f_ac40f6d9142b,0.663293,0.873349,29.469606
4,object,PC 19960097,PC 20130107,30d266ad-d438-495d-304f-db750bb1066a,3e533bae-6641-4aa5-89b2-5f43a8408184,30d266ad_d438_495d_304f_db750bb1066a,3e533bae_6641_4aa5_89b2_5f43a8408184,0.665474,0.073178,45.298528
5,object,PC 19960097,PC 19840074,30d266ad-d438-495d-304f-db750bb1066a,c20e3256-c083-444f-b709-5fe5db3b5fc9,30d266ad_d438_495d_304f_db750bb1066a,c20e3256_c083_444f_b709_5fe5db3b5fc9,0.582131,0.031738,52.794768
6,object,PC 19960097,PC 20080147,30d266ad-d438-495d-304f-db750bb1066a,46662bf5-37b7-4ae1-e994-32ea57c282de,30d266ad_d438_495d_304f_db750bb1066a,46662bf5_37b7_4ae1_e994_32ea57c282de,0.601663,0.026691,51.333103
7,object,PC 19960097,PC 20070296,30d266ad-d438-495d-304f-db750bb1066a,654e58f8-476a-485c-e30f-ef12fd9c7c3c,30d266ad_d438_495d_304f_db750bb1066a,654e58f8_476a_485c_e30f_ef12fd9c7c3c,0.587040,0.045883,52.119131
8,object,PC 19960097,PC 19720166,30d266ad-d438-495d-304f-db750bb1066a,1375e361-cf97-4c48-f545-0fb811bab781,30d266ad_d438_495d_304f_db750bb1066a,1375e361_cf97_4c48_f545_0fb811bab781,0.592559,0.022584,52.143619
9,object,PC 19960097,pc 20230077,30d266ad-d438-495d-304f-db750bb1066a,609ac9bc-13f5-4c61-8063-2567cbe6157a,30d266ad_d438_495d_304f_db750bb1066a,609ac9bc_13f5_4c61_8063_2567cbe6157a,0.452571,0.052428,62.745747


In [6]:

def percentile_df_sim_weights(df_sim):
    """Makes percentiles for df_sim weights for each source"""
    df_sim['source_max_weight'] = float(0.0)
    df_sim['source_min_weight'] = float(0.0)
    quantiles = [0.9, .75, 0.5, .25, 0.1,]
    q_cols = {f'source_{int(q * 100)}_weight': q for q in quantiles}
    for col, _ in q_cols.items():
        df_sim[col] = float(0.0)
    for source in df_sim['source'].unique():
        act_index = df_sim['source'] == source
        df_sim.loc[act_index, 'source_max_weight'] = df_sim[act_index]['weight'].max()
        df_sim.loc[act_index, 'source_min_weight'] = df_sim[act_index]['weight'].max()
        for col, q in q_cols.items():
            df_sim.loc[act_index, col] = df_sim[act_index]['weight'].quantile(q)
    return df_sim

def select_min_weights_by_col(df_sim, col_criteria):
    df_sim['use_weight'] = False
    for i, row in df_sim.iterrows():
        df_sim.at[i, 'use_weight'] = (row['weight'] >= row[col_criteria])
    return df_sim

df_sim = percentile_df_sim_weights(df_sim)
# df_sim = select_min_weights_by_col(df_sim, col_criteria='source_90_weight')
df_sim = select_min_weights_by_col(df_sim, col_criteria='source_max_weight')


In [7]:
act_index = df_sim['weight'] >= df_sim['source_90_weight']
print(f'Make force directed graph with {len(df_sim[act_index].index)} edges')

Make force directed graph with 2080 edges


In [8]:
from pyvis.network import Network

net = Network(
    height='750px', 
    width='100%', 
    bgcolor='#222222', 
    font_color='white', 
    notebook=True, 
    cdn_resources='in_line'
)
# 3. Add unique nodes with image properties
all_nodes = df_sim[act_index]['source'].unique().tolist()
all_nodes += df_sim[act_index]['target'].unique().tolist()
unique_nodes = set(all_nodes)
for uuid_g in unique_nodes:
    uuid = uuid_g.replace('_', '-')
    uuid_index = df_sample['uuid'] == uuid
    label = df_sample[uuid_index]['label'].iloc[0]
    thumb_uri = df_sample[uuid_index]['thumbnail_uri'].iloc[0]
    if str(thumb_uri).startswith('http'):
        net.add_node(
            uuid_g, 
            label=label, 
            shape='image', 
            image=thumb_uri,
            size=25,
            x=float(df_sample[uuid_index]['longitude'].iloc[0]),
            y=float(df_sample[uuid_index]['latitude'].iloc[0]),
            # physics=False,
        )
    else:
        net.add_node(
            uuid_g, 
            label=label, 
            shape='dot', 
            size=20,
            x=float(df_sample[uuid_index]['longitude'].iloc[0]),
            y=float(df_sample[uuid_index]['latitude'].iloc[0]),
            # physics=False,
        )

# 4. Add edges
for _, row in df_sim[act_index].iterrows():
    net.add_edge(row['source'], row['target'], weight=(row['weight']))

# 5. Generate and open the graph
net.toggle_physics(True)
net.prep_notebook()
# net.show(net_output_html, notebook=False)
net.save_graph(net_output_html)

from IPython.display import IFrame
from IPython.display import display, HTML

HTML(filename=net_output_html)
# 1. Write the standalone HTML file to disk
# net.write_html(net_output_html)

# 2. Force it to load manually inside a cell window
# IFrame(src=net_output_html, width="100%", height="750px")


# OLD CODE
from d3blocks import D3Blocks

d3 = D3Blocks()
d3.elasticgraph(df=df_sim[act_index], filepath=output_html, charge=2500)
for uuid_g, node_dict in d3.Elasticgraph.D3graph.node_properties.items():
    uuid = uuid_g.replace('_', '-')
    act_index = df_sample['uuid'] == uuid
    if df_sample[act_index].empty:
        print(f'Cannot update {uuid_g} from {uuid}')
        print(node_dict)
        continue
    item_class = str(df_sample[act_index]['item_class__slug'].iloc[0])
    item_class = item_class.split('-cat-')[-1]
    label = df_sample[act_index]['label'].iloc[0]
    # print(f'Group {item_class} label {label}')
    # d3.Elasticgraph.D3graph.node_properties[uuid_g]['group'] = item_class
    d3.Elasticgraph.D3graph.node_properties[uuid_g]['label'] = label
    d3.Elasticgraph.D3graph.node_properties[uuid_g]['fontsize'] = 6
    
for s_t_tup, edge_dict in d3.Elasticgraph.D3graph.edge_properties.items():
    source = s_t_tup[0]
    target = s_t_tup[1]
    weight = float(edge_dict['weight'])
    edge_w = round((3 * weight), 2)
    d3.Elasticgraph.D3graph.edge_properties[(source, target)]['edge_size'] = edge_w

html = d3.Elasticgraph.show(filepath=output_html)


